# BraTS post processing

In [1]:
import os

# Path to nnUNet labels for the test split (set up by the user outside the repo)
DIR_DATA  = "../../../../../data/nnUNet_raw_data/Task182_BraTS2020_80_20/labelsTs/"
DIR_PREDS = "../../../../../data/BraTS_seg_uncertain_predictions"

gt_files = sorted(f for f in os.listdir(DIR_DATA)  if f.endswith('.nii.gz'))
pred_npz  = sorted(f for f in os.listdir(DIR_PREDS) if f.endswith('.npz'))
case_ids  = [f.replace('.npz', '') for f in pred_npz]

print(f"Ground truth files : {len(gt_files)}")
print(f"Prediction cases   : {len(case_ids)}")
print(f"First case id      : {case_ids[0]}")

Ground truth files : 73
Prediction cases   : 73
First case id      : BraTS20_Training_002


## Whole Tumour (WT) extraction

The model is a multi-region nnUNet ensemble trained on BraTS 2020. Each `.npz` file
contains a `softmax` array of shape `(3, Z, Y, X)` in nnUNet's internal axis order
(slice-first), with one sigmoid channel per region:

| Channel | Region |
|---------|--------|
| 0 | **Whole Tumour (WT)** — all non-background voxels |
| 1 | Tumour Core (TC) |
| 2 | Enhancing Tumour (ET) |

The corresponding `.nii.gz` file stores the post-processed multi-class prediction in
NIfTI axis order `(X, Y, Z)`. Label `> 0` recovers the WT binary mask.

Ground-truth labels follow the original BraTS convention (0 = background, 1 = NCR/NET,
2 = oedema, 4 = enhancing); WT ground truth = `gt > 0`.

Each 3-D volume is kept intact. The output arrays have shape `(N, 1, Z, Y, X)`,
where N is the number of cases and the channel dimension (size 1) holds the WT region.

In [2]:
import numpy as np
import nibabel as nib
from tqdm import tqdm

all_p_hat, all_y_hat, all_y = [], [], []

for case_id in tqdm(case_ids, desc="Processing cases"):
    # --- softmax: (3, Z, Y, X) nnUNet axis order ---
    softmax = np.load(os.path.join(DIR_PREDS, case_id + '.npz'))['softmax'].astype(np.float32)
    # Channel 0 = WT probability; already in (Z, Y, X) order
    p_hat_vol = softmax[0]                                    # (Z, Y, X)

    # --- binarised prediction: NIfTI (X, Y, Z); WT = any label > 0 ---
    pred_arr = np.array(
        nib.load(os.path.join(DIR_PREDS, case_id + '.nii.gz')).dataobj,
        dtype=np.uint8,
    )
    y_hat_vol = (pred_arr > 0).astype(np.float32).transpose(2, 1, 0)  # (Z, Y, X)

    # --- ground truth: NIfTI (X, Y, Z); BraTS labels {0,1,2,4}; WT = any label > 0 ---
    gt_arr = np.array(
        nib.load(os.path.join(DIR_DATA, case_id + '.nii.gz')).dataobj,
        dtype=np.uint8,
    )
    y_vol = (gt_arr > 0).astype(np.float32).transpose(2, 1, 0)        # (Z, Y, X)

    # Add channel dimension: (1, Z, Y, X)
    all_p_hat.append(p_hat_vol[np.newaxis])
    all_y_hat.append(y_hat_vol[np.newaxis])
    all_y.append(y_vol[np.newaxis])

p_hat = np.stack(all_p_hat)   # (N, 1, Z, Y, X)
y_hat = np.stack(all_y_hat)   # (N, 1, Z, Y, X)
y     = np.stack(all_y)       # (N, 1, Z, Y, X)

print(f"Cases processed : {len(all_p_hat)}")
print(f"p_hat shape     : {p_hat.shape}")
print(f"y_hat shape     : {y_hat.shape}")
print(f"y shape         : {y.shape}")

Processing cases: 100%|██████████| 73/73 [00:31<00:00,  2.31it/s]


Cases processed : 73
p_hat shape     : (73, 1, 155, 240, 240)
y_hat shape     : (73, 1, 155, 240, 240)
y shape         : (73, 1, 155, 240, 240)


In [3]:
output_path = os.path.join('..', '..', 'medical-imaging', 'data', 'brats.npz')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
np.savez_compressed(output_path, p_hat=p_hat, y_hat=y_hat, y=y)
print(f"Saved → {output_path}")

Saved → ../../medical-imaging/data/brats.npz


## Sanity check

Verify the saved file and report dataset statistics.

In [4]:
data = np.load(output_path)
p_hat_v, y_hat_v, y_v = data['p_hat'], data['y_hat'], data['y']

tumor_vols = np.any(y_v > 0, axis=(-3, -2, -1))
print(f"Cases total              : {len(y_v)}")
print(f"Cases with tumour        : {tumor_vols.sum()} ({100*tumor_vols.mean():.1f} %)")
print(f"p_hat range              : [{p_hat_v.min():.4f}, {p_hat_v.max():.4f}]")

# Dice score per volume (WT)
def dice(a, b):
    inter = (a * b).sum()
    return 2 * inter / (a.sum() + b.sum() + 1e-8)

dice_scores = [dice(y_hat_v[i, 0], y_v[i, 0]) for i in range(len(y_v))]
print(f"Mean Dice (WT, all volumes) : {np.mean(dice_scores):.4f}")

Cases total              : 73
Cases with tumour        : 73 (100.0 %)
p_hat range              : [0.0000, 1.0000]
Mean Dice (WT, all volumes) : 0.9123
